# Ejercicio 9: Uso de la API de Google Gemini

En este ejercicio vamos a aprender a utilizar la API de OpenAI

## 1. Uso básico

El siguiente código sirve para conectarse con la API de Google Gemini de forma básica

In [3]:
from google import genai
from kaggle_secrets import UserSecretsClient

# 1. Configuración de la clave
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("api-key-ir")

# 2. Inicializar cliente
client = genai.Client(api_key=api_key)

In [4]:
# 3. Llamada con el nombre de modelo completo (models/...)
try:
    response = client.models.generate_content(
        model="models/gemini-2.5-flash-lite", 
        contents="Explica brevemente cómo funciona la IA."
    )
    print(response.text)
except Exception as e:
    print(f"Error: {e}")

La Inteligencia Artificial (IA) funciona, en términos generales, **simulando procesos de inteligencia humana en máquinas**. Aquí te explico los conceptos clave de forma breve:

1.  **Datos:** La IA se alimenta de enormes cantidades de datos. Piensa en esto como la "experiencia" o el "aprendizaje" para la IA. Estos datos pueden ser imágenes, texto, números, sonidos, etc.

2.  **Algoritmos y Modelos:** Los datos se procesan mediante algoritmos, que son conjuntos de reglas o instrucciones. Estos algoritmos crean y entrenan "modelos". Un modelo es, en esencia, un sistema matemático complejo que ha aprendido a identificar patrones, relaciones o tomar decisiones a partir de los datos.

3.  **Aprendizaje (Machine Learning):** La mayoría de la IA moderna se basa en el "aprendizaje automático". Hay varios tipos:
    *   **Aprendizaje Supervisado:** Se le dan datos con "respuestas correctas" (etiquetas) para que el modelo aprenda a predecir esas respuestas. (Ej: mostrarle miles de fotos de perro

## 2. Retrieval

### 2.1 Cargo el corpus de 20 News Groups

In [1]:
from sklearn.datasets import fetch_20newsgroups

newsgroups = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
newsgroupsdocs = newsgroups.data

In [2]:
import pandas as pd

df = pd.DataFrame(newsgroupsdocs, columns=['text'])
df

,text
0,\n\nI am sure some bashers of Pens fans are pr...
1,My brother is in the market for a high-perform...
2,\n\n\n\n\tFinally you said what you dream abou...
3,\nThink!\n\nIt's the SCSI card doing the DMA t...
4,1) I have an old Jasmine drive which I cann...
...,...
18841,DN> From: nyeda@cnsvax.uwec.edu (David Nye)\nD...
18842,\nNot in isolated ground recepticles (usually ...
18843,I just installed a DX2-66 CPU in a clone mothe...
18844,\nWouldn't this require a hyper-sphere. In 3-...


### 2.2 Transformo a embeddings

In [5]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

# df = df.dropna(subset=["text"]).reset_index(drop=True)

# Limpieza básica
def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_norm"] = df["text"].astype(str).map(normalize_text)

df.head()

,text,text_norm
0,\n\nI am sure some bashers of Pens fans are pr...,I am sure some bashers of Pens fans are pretty...
1,My brother is in the market for a high-perform...,My brother is in the market for a high-perform...
2,\n\n\n\n\tFinally you said what you dream abou...,Finally you said what you dream about. Mediter...
3,\nThink!\n\nIt's the SCSI card doing the DMA t...,Think! It's the SCSI card doing the DMA transf...
4,1) I have an old Jasmine drive which I cann...,1) I have an old Jasmine drive which I cannot ...


In [6]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100):
    """
    Chunking por caracteres.
    max_chars ~ 600-1000 suele funcionar bien.
    overlap ayuda a no cortar ideas a la mitad.
    """
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end]
        chunk = chunk.strip()
        if len(chunk) > 0:
            chunks.append(chunk)
        if end == n:
            break
        start = max(0, end - overlap)
    return chunks

records = []
for i, row in df.iterrows():
    chunks = chunk_text(row["text_norm"], max_chars=800, overlap=100)
    for j, ch in enumerate(chunks):
        records.append({
            "doc_id": int(i),
            "chunk_id": j,
            "text": ch
        })

chunks_df = pd.DataFrame(records)
chunks_df.head(), len(chunks_df)

(   doc_id  chunk_id                                               text
 0       0         0  I am sure some bashers of Pens fans are pretty...
 1       1         0  My brother is in the market for a high-perform...
 2       2         0  Finally you said what you dream about. Mediter...
 3       2         1  urds and Turks once upon a time! Ohhhh so swed...
 4       3         0  Think! It's the SCSI card doing the DMA transf...,
 38871)

In [7]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-base-v2"   # recomendado para retrieval
model = SentenceTransformer(MODEL_NAME)

# Textos a indexar (pasajes)
passages = ["passage: " + t for t in chunks_df["text"].tolist()]

2026-01-09 02:59:11.617085: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767927551.810582      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767927551.864256      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767927552.316597      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767927552.316632      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767927552.316635      55 computation_placer.cc:177] computation placer alr

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [8]:
# Embeddings (N x D)
# Se debe usar normalize_embeddings=True para similitud coseno
embeddings = model.encode(
    passages,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

Batches:   0%|          | 0/2430 [00:00<?, ?it/s]

In [9]:
print(embeddings.shape, embeddings.dtype)

(38871, 768) float32


In [10]:
def embed_query(query: str) -> np.ndarray:
    q = "query: " + query
    vec = model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    return vec

query_text = "Soccer matches"

query_vec = embed_query(query_text)
query_vec.shape

(1, 768)

### 2.3 Creo una query y hago la búsqueda

Obtengo los 5 documentos más similares a mi query

In [ ]:
# query es un string y tiene que pasar a ser un embedding de 700+ dimensiones

# Transformo el corpus a embeddings, de 700+ dimensiones

# Se hace búsqueda por similitud (Ej. FAISS), entre el query para todos los demás. Obtengo los 5 elementos más relevantes (top5), son texto, no embeddings.

# API GEMINI
# Enviar el top5 al modelo de lenguaje, el prompt va a ser: "la query fue esto... y los resultados son estos..., dame un resumen". 
# Al usuario le doy la respuesta como un texto.

In [14]:
# Cálculo de similitud (producto punto / coseno)
similarity_values = embeddings @ query_vec.T
similarity_values = similarity_values.ravel()

# Selección de los mejores resultados
num_results = 5
best_positions = np.argsort(similarity_values)[-num_results:][::-1]

print(f"Consulta procesada correctamente: \"{query_text}\"")

Consulta procesada correctamente: "Soccer matches"


In [15]:
print("\nFragmentos más relevantes encontrados:\n")

for rank, position in enumerate(best_positions, start=1):
    sim_score = similarity_values[position]
    fragment = chunks_df.loc[position, "text"]
    document_id = chunks_df.loc[position, "doc_id"]

    print(f"{rank}. Documento {document_id} | Similitud = {sim_score:.4f}")
    print(f"Contenido:\n{fragment}")
    print("=" * 80)


Fragmentos más relevantes encontrados:

1. Documento 5715 | Similitud = 0.8140
Contenido:
Is there any games being shown here in the US from the WC??? Thanks
2. Documento 4554 | Similitud = 0.8135
Contenido:
#5 20:00 April 30: Semifinals A #1/B #4 - A #3/B #2 15:30 A #4/B #1 - A #2/B #3 20:00 May 1: Relegation 14:30 Bronze medal game 19:00 May 2: FINAL 15:00
3. Documento 10960 | Similitud = 0.8130
Contenido:
27: Quarterfinals Sweden - USA 15:30 Russia - Germany 20:00 April 28: Quarterfinals Canada - Finland 15:30 Italy/Switzerland - Czech republic 20:00 April 29: Relegation A #5 - B #6 15:30 A #6 - B #5 20:00 April 30: Semifinals A #1/B #4 - A #3/B #2 15:30 A #4/B #1 - A #2/B #3 20:00 May 1: Relegation 14:30 Bronze medal game 19:00 May 2: FINAL 15:00
4. Documento 556 | Similitud = 0.8129
Contenido:
#2 20:00 April 28: Quarterfinals A #1 - B #4 15:30 A #4 - B #1 20:00 April 29: Relegation A #5 - B #6 15:30 A #6 - B #5 20:00 April 30: Semifinals A #1/B #4 - A #3/B #2 15:30 A #4/B #1 - A 

In [16]:
# Construcción del contexto para el modelo
retrieved_texts = chunks_df.loc[best_positions, "text"].to_list()
assembled_context = "\n\n".join(retrieved_texts)

assembled_context

'Is there any games being shown here in the US from the WC??? Thanks\n\n#5 20:00 April 30: Semifinals A #1/B #4 - A #3/B #2 15:30 A #4/B #1 - A #2/B #3 20:00 May 1: Relegation 14:30 Bronze medal game 19:00 May 2: FINAL 15:00\n\n27: Quarterfinals Sweden - USA 15:30 Russia - Germany 20:00 April 28: Quarterfinals Canada - Finland 15:30 Italy/Switzerland - Czech republic 20:00 April 29: Relegation A #5 - B #6 15:30 A #6 - B #5 20:00 April 30: Semifinals A #1/B #4 - A #3/B #2 15:30 A #4/B #1 - A #2/B #3 20:00 May 1: Relegation 14:30 Bronze medal game 19:00 May 2: FINAL 15:00\n\n#2 20:00 April 28: Quarterfinals A #1 - B #4 15:30 A #4 - B #1 20:00 April 29: Relegation A #5 - B #6 15:30 A #6 - B #5 20:00 April 30: Semifinals A #1/B #4 - A #3/B #2 15:30 A #4/B #1 - A #2/B #3 20:00 May 1: Relegation 14:30 Bronze medal game 19:00 May 2: FINAL 15:00\n\nA #2 - B #3 15:30 A #3 - B #2 20:00 April 28: Quarterfinals A #1 - B #4 15:30 A #4 - B #1 20:00 April 29: Relegation A #5 - B #6 15:30 A #6 - B #5 

In [17]:
# Creación del prompt
llm_prompt = f"""
Actúas como un asistente especializado en análisis de información.
A partir del contexto proporcionado, elabora un resumen claro y preciso.
Si el contexto no permite responder adecuadamente, indícalo explícitamente.

INFORMACIÓN DISPONIBLE:
{assembled_context}

CONSULTA:
{query_text}

RESPUESTA:
"""

In [18]:
llm_response = client.models.generate_content(
    model="models/gemini-2.5-flash-lite",
    contents=llm_prompt
)

print("\n--- SALIDA DEL MODELO GEMINI ---")
print(llm_response.text)


--- SALIDA DEL MODELO GEMINI ---
El contexto proporcionado detalla un calendario de partidos, incluyendo cuartos de final, semifinales, partido por el bronce y la final. Sin embargo, **no se especifica si estos partidos son de fútbol ("soccer") ni si se muestran en Estados Unidos**.

La información disponible se centra en las fechas, horas y las eliminatorias o enfrentamientos entre equipos (representados por A# y B#), así como algunos nombres de países como "Sweden", "USA", "Russia", "Germany", "Canada", "Finland", "Italy/Switzerland" y "Czech republic".
